# Poker Arena - Colab CUDA Deep CFR training

This notebook stages the current `poker_bot` source from Google Drive onto Colab's local SSD, verifies that the post-fix reservoir and frontier-chunking code is present, trains the mixed 3-9 player 100BB policy, validates the result, and downloads a small engine-ready bundle automatically.

Before choosing **Runtime > Run all**:

1. Upload the complete project folder as `MyDrive/poker_bot`.
2. Select a GPU runtime in Colab. A T4 is supported.
3. Keep this tab open while training. If Colab disconnects, run the notebook again; it resumes from the latest Drive snapshot.

The default is the full 100-iteration workload: 4,096 traversals per possible seat, randomized 3-9 player sessions, and 100BB stacks. A T4 is likely to need substantially longer than a few hours. The resumable training snapshot remains in Drive because it can be several gigabytes; the automatically downloaded ZIP contains the much smaller `.pt` inference policy required by the poker engine, its summary, integrity report, and deployment instructions.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

# Upload the project to this exact Drive location for zero-edit Run All operation.
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/poker_bot')
LOCAL_PROJECT_DIR = Path('/content/poker_bot_colab')
RUN_NAME = 'cuda_deep_cfr_mixed_3to9_100bb_colab_formal_100'

# Formal workload. The target is an absolute final iteration, including resumes.
TARGET_ITERATIONS = 100
TRAVERSALS_PER_PLAYER = 4096
MIN_PLAYERS = 3
MAX_PLAYERS = 9
SMALL_BLIND = 5
BIG_BLIND = 10
STARTING_STACK = 1000  # 100 big blinds
ADVANTAGE_CAPACITY = 300_000
STRATEGY_CAPACITY = 300_000
HIDDEN = '512,512,256'
ADVANTAGE_TRAIN_STEPS = 1000
STRATEGY_TRAIN_STEPS = 4000
BATCH_SIZE = 8192
SNAPSHOT_EVERY = 10
RANDOM_SEED = 17

if not DRIVE_PROJECT_DIR.is_dir():
    raise FileNotFoundError(
        f'Project folder not found at {DRIVE_PROJECT_DIR}. '
        'Upload/rename it to MyDrive/poker_bot, then run all again.'
    )
print(f'Using Drive project: {DRIVE_PROJECT_DIR}')


## Stage only the training source

Old runs, virtual environments, legacy checkpoints, and Git metadata are deliberately not copied to the Colab VM. Training reads and writes locally for speed, while resumable artifacts are synchronized separately to Drive. Colab's preinstalled CUDA-compatible PyTorch is retained to avoid replacing it with a mismatched wheel.

In [ ]:
import json
import os
import shutil
import subprocess
import sys

if sys.version_info < (3, 11):
    raise RuntimeError(f'Python 3.11+ is required; found {sys.version.split()[0]}')

if LOCAL_PROJECT_DIR.exists():
    shutil.rmtree(LOCAL_PROJECT_DIR)

ignored_names = shutil.ignore_patterns(
    '.git', '.venv', '.pytest_cache', '__pycache__', '__MACOSX',
    '*.egg-info', '*.pyc', 'runs', 'legacy'
)
shutil.copytree(DRIVE_PROJECT_DIR, LOCAL_PROJECT_DIR, ignore=ignored_names)

trainer_source = LOCAL_PROJECT_DIR / 'src/poker_arena/cfr/cuda_deep_cfr.py'
cli_source = LOCAL_PROJECT_DIR / 'examples/train_cuda_deep_cfr.py'
if not trainer_source.is_file() or not cli_source.is_file():
    raise FileNotFoundError('The uploaded folder is missing the CUDA Deep CFR trainer sources.')
source_text = trainer_source.read_text(encoding='utf-8')
required_fix_markers = ('index_copy_', 'index_fill_', 'frontier_chunk_splits', 'minimum_players')
missing_markers = [marker for marker in required_fix_markers if marker not in source_text]
if missing_markers:
    raise RuntimeError(
        'Uploaded source predates required training fixes; missing markers: '
        + ', '.join(missing_markers)
    )

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps', '-e', str(LOCAL_PROJECT_DIR)
])
os.chdir(LOCAL_PROJECT_DIR)
if str(LOCAL_PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(LOCAL_PROJECT_DIR))
if str(LOCAL_PROJECT_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(LOCAL_PROJECT_DIR / 'src'))
print('Training-only project environment is ready on the local SSD.')


## Verify and size the GPU

Parallel traversal count and frontier budget are selected from available VRAM. The T4 profile uses 256 parallel sessions with conservative frontier chunking. This raises utilization without allowing tree frontiers to consume all 16 GB.

In [ ]:
import platform
import torch

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. In Colab select Runtime > Change runtime type > GPU, then run all again.')
if torch.cuda.device_count() != 1:
    raise RuntimeError(f'Expected exactly one visible GPU, found {torch.cuda.device_count()}.')

gpu = torch.cuda.get_device_properties(0)
GPU_NAME = gpu.name
GPU_VRAM_GIB = gpu.total_memory / 1024**3

if GPU_VRAM_GIB <= 12.5:
    PARALLEL_TRAVERSALS, MAX_FRONTIER_ROWS = 160, 98_304
elif GPU_VRAM_GIB <= 16.5:
    PARALLEL_TRAVERSALS, MAX_FRONTIER_ROWS = 256, 131_072
elif GPU_VRAM_GIB <= 25:
    PARALLEL_TRAVERSALS, MAX_FRONTIER_ROWS = 384, 196_608
else:
    PARALLEL_TRAVERSALS, MAX_FRONTIER_ROWS = 512, 262_144

gpu_info = {
    'python': platform.python_version(),
    'torch': torch.__version__,
    'torch_cuda': torch.version.cuda,
    'gpu': GPU_NAME,
    'compute_capability': list(torch.cuda.get_device_capability(0)),
    'vram_gib': round(GPU_VRAM_GIB, 2),
    'parallel_traversals': PARALLEL_TRAVERSALS,
    'max_frontier_rows': MAX_FRONTIER_ROWS,
}
print(json.dumps(gpu_info, indent=2))
subprocess.run(['nvidia-smi'], check=False)


## Prepare a fresh run or resume safely

The run name is isolated from old invalid checkpoints. Every tenth completed iteration, the trainer creates an atomic local snapshot and this notebook copies it atomically into `MyDrive/poker_bot/colab_runs/<run name>/`. Rerunning the notebook adds only the iterations needed to reach the absolute target of 100.

In [ ]:
DRIVE_RUN_DIR = DRIVE_PROJECT_DIR / 'colab_runs' / RUN_NAME
LOCAL_RUN_DIR = LOCAL_PROJECT_DIR / 'runs' / RUN_NAME
DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_RUN_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_SNAPSHOT = LOCAL_RUN_DIR / f'{RUN_NAME}_snapshot.pt'
LOCAL_POLICY = LOCAL_RUN_DIR / f'{RUN_NAME}_policy.pt'
LOCAL_SUMMARY = LOCAL_RUN_DIR / f'{RUN_NAME}_summary.json'
LOCAL_LOG = LOCAL_RUN_DIR / f'{RUN_NAME}_training.log'
DRIVE_SNAPSHOT = DRIVE_RUN_DIR / LOCAL_SNAPSHOT.name
DRIVE_POLICY = DRIVE_RUN_DIR / LOCAL_POLICY.name
DRIVE_SUMMARY = DRIVE_RUN_DIR / LOCAL_SUMMARY.name
DRIVE_LOG = DRIVE_RUN_DIR / LOCAL_LOG.name
DRIVE_PROGRESS = DRIVE_RUN_DIR / 'progress.json'

run_config = {
    'run_name': RUN_NAME,
    'target_iterations': TARGET_ITERATIONS,
    'traversals_per_player': TRAVERSALS_PER_PLAYER,
    'min_players': MIN_PLAYERS,
    'max_players': MAX_PLAYERS,
    'starting_stack': STARTING_STACK,
    'small_blind': SMALL_BLIND,
    'big_blind': BIG_BLIND,
    'advantage_capacity': ADVANTAGE_CAPACITY,
    'strategy_capacity': STRATEGY_CAPACITY,
    'hidden': HIDDEN,
    'advantage_train_steps': ADVANTAGE_TRAIN_STEPS,
    'strategy_train_steps': STRATEGY_TRAIN_STEPS,
    'batch_size': BATCH_SIZE,
    'random_seed': RANDOM_SEED,
    'gpu': gpu_info,
}

completed_iterations = 0
resume_from_drive = False
if DRIVE_PROGRESS.is_file():
    progress = json.loads(DRIVE_PROGRESS.read_text(encoding='utf-8'))
    completed_iterations = int(progress.get('completed_iterations', 0))
    saved_config = progress.get('run_config', {})
    immutable_fields = (
        'traversals_per_player', 'min_players', 'max_players', 'starting_stack',
        'small_blind', 'big_blind', 'advantage_capacity', 'strategy_capacity',
        'hidden', 'advantage_train_steps', 'strategy_train_steps', 'batch_size', 'random_seed'
    )
    mismatches = {
        key: (saved_config.get(key), run_config.get(key))
        for key in immutable_fields if saved_config.get(key) != run_config.get(key)
    }
    if mismatches:
        raise RuntimeError(f'Existing Drive run has incompatible settings: {mismatches}. Change RUN_NAME for a fresh run.')
    resume_from_drive = 0 < completed_iterations < TARGET_ITERATIONS and DRIVE_SNAPSHOT.is_file()

already_complete = (
    completed_iterations >= TARGET_ITERATIONS
    and DRIVE_POLICY.is_file()
    and DRIVE_SUMMARY.is_file()
)

if already_complete:
    shutil.copy2(DRIVE_POLICY, LOCAL_POLICY)
    shutil.copy2(DRIVE_SUMMARY, LOCAL_SUMMARY)
    if DRIVE_LOG.is_file():
        shutil.copy2(DRIVE_LOG, LOCAL_LOG)
    print(f'Run is already complete at iteration {completed_iterations}; training will be skipped.')
elif resume_from_drive:
    print(f'Copying iteration-{completed_iterations} resume snapshot from Drive to the local SSD...')
    shutil.copy2(DRIVE_SNAPSHOT, LOCAL_SNAPSHOT)
    if DRIVE_LOG.is_file():
        shutil.copy2(DRIVE_LOG, LOCAL_LOG)
    print('Resume snapshot is ready.')
else:
    if completed_iterations:
        raise RuntimeError('Drive progress exists but its resumable snapshot is missing.')
    print('No compatible resume found; starting a fresh post-fix run at iteration 0.')


## Train

This is the long-running cell. Each progress line is also appended to a log. Snapshot uploads can take several minutes because the resumable state includes all nine seats' networks and reservoirs.

In [ ]:
import re
import time

def atomic_copy_to_drive(source: Path, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(f'.{destination.name}.uploading')
    temporary.unlink(missing_ok=True)
    shutil.copy2(source, temporary)
    os.replace(temporary, destination)

def write_drive_progress(done: int, status: str) -> None:
    payload = {
        'completed_iterations': done,
        'target_iterations': TARGET_ITERATIONS,
        'status': status,
        'updated_unix': time.time(),
        'run_config': run_config,
    }
    temporary = DRIVE_PROGRESS.with_name('.progress.json.uploading')
    temporary.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    os.replace(temporary, DRIVE_PROGRESS)

def sync_snapshot(done: int) -> None:
    if not LOCAL_SNAPSHOT.is_file():
        raise FileNotFoundError(f'Trainer announced a snapshot, but {LOCAL_SNAPSHOT} is missing.')
    size_gib = LOCAL_SNAPSHOT.stat().st_size / 1024**3
    print(f'Uploading {size_gib:.2f} GiB iteration-{done} resume snapshot to Drive...', flush=True)
    atomic_copy_to_drive(LOCAL_SNAPSHOT, DRIVE_SNAPSHOT)
    write_drive_progress(done, 'training')
    if LOCAL_LOG.is_file():
        atomic_copy_to_drive(LOCAL_LOG, DRIVE_LOG)
    print('Drive resume point updated.', flush=True)

if not already_complete:
    remaining_iterations = TARGET_ITERATIONS - completed_iterations
    command = [
        sys.executable, '-u', '-B', 'examples/train_cuda_deep_cfr.py',
        '--iterations', str(remaining_iterations),
        '--traversals-per-player', str(TRAVERSALS_PER_PLAYER),
        '--parallel-traversals', str(PARALLEL_TRAVERSALS),
        '--max-frontier-rows', str(MAX_FRONTIER_ROWS),
        '--seats', str(MAX_PLAYERS),
        '--min-players', str(MIN_PLAYERS),
        '--small-blind', str(SMALL_BLIND),
        '--big-blind', str(BIG_BLIND),
        '--starting-stack', str(STARTING_STACK),
        '--advantage-capacity', str(ADVANTAGE_CAPACITY),
        '--strategy-capacity', str(STRATEGY_CAPACITY),
        '--hidden', HIDDEN,
        '--advantage-train-steps', str(ADVANTAGE_TRAIN_STEPS),
        '--strategy-train-steps', str(STRATEGY_TRAIN_STEPS),
        '--batch-size', str(BATCH_SIZE),
        '--random-seed', str(RANDOM_SEED),
        '--gpus', '0',
        '--snapshot-out', str(LOCAL_SNAPSHOT),
        '--snapshot-every', str(SNAPSHOT_EVERY),
        '--policy-out', str(LOCAL_POLICY),
        '--summary-out', str(LOCAL_SUMMARY),
        '--log-every', '1',
    ]
    if resume_from_drive:
        command.extend(['--resume-snapshot', str(LOCAL_SNAPSHOT)])

    print('Launching:', ' '.join(command), flush=True)
    process_env = os.environ.copy()
    process_env['PYTHONUNBUFFERED'] = '1'
    process_env['PYTHONDONTWRITEBYTECODE'] = '1'
    snapshot_pattern = re.compile(r'refreshed resumable snapshot at iteration (\d+)')

    with LOCAL_LOG.open('a', encoding='utf-8', buffering=1) as log_handle:
        process = subprocess.Popen(
            command, cwd=LOCAL_PROJECT_DIR, env=process_env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='', flush=True)
            log_handle.write(line)
            match = snapshot_pattern.search(line)
            if match:
                log_handle.flush()
                sync_snapshot(int(match.group(1)))
        return_code = process.wait()

    if return_code != 0:
        if LOCAL_LOG.is_file():
            atomic_copy_to_drive(LOCAL_LOG, DRIVE_LOG)
        raise RuntimeError(f'CUDA Deep CFR training exited with code {return_code}. See {DRIVE_LOG}.')

    for artifact, destination in (
        (LOCAL_SNAPSHOT, DRIVE_SNAPSHOT),
        (LOCAL_POLICY, DRIVE_POLICY),
        (LOCAL_SUMMARY, DRIVE_SUMMARY),
        (LOCAL_LOG, DRIVE_LOG),
    ):
        if not artifact.is_file():
            raise FileNotFoundError(f'Expected completed artifact is missing: {artifact}')
        print(f'Copying final {artifact.name} to Drive...', flush=True)
        atomic_copy_to_drive(artifact, destination)
    write_drive_progress(TARGET_ITERATIONS, 'completed_pending_validation')
    completed_iterations = TARGET_ITERATIONS

print(f'Training artifacts are ready at iteration {completed_iterations}.')


## Integrity gates and engine smoke tests

The engine bundle is created only if the summary is fresh, every table size from 3 through 9 has traversal coverage, depth-limit fallback was never needed, all learned strategy weights are finite, and the policy can choose and apply a legal action at every supported table size.

In [ ]:
import hashlib

if not LOCAL_POLICY.is_file() or not LOCAL_SUMMARY.is_file():
    raise FileNotFoundError('Completed policy/summary is missing; training did not finish successfully.')

summary = json.loads(LOCAL_SUMMARY.read_text(encoding='utf-8'))
failures = []
expected_player_counts = set(range(MIN_PLAYERS, MAX_PLAYERS + 1))
coverage = {int(key): int(value) for key, value in summary.get('player_count_traversals', {}).items()}

if int(summary.get('iterations', -1)) != TARGET_ITERATIONS:
    failures.append(f"iterations={summary.get('iterations')} (expected {TARGET_ITERATIONS})")
if int(summary.get('depth_limit_rollouts', -1)) != 0:
    failures.append(f"depth_limit_rollouts={summary.get('depth_limit_rollouts')}")
if summary.get('latest_iteration_retained') is not True:
    failures.append('latest_iteration_retained is not true')
if set(coverage) != expected_player_counts or any(coverage.get(count, 0) <= 0 for count in expected_player_counts):
    failures.append(f'player-count coverage is incomplete: {coverage}')
if summary.get('supported_player_counts') != list(range(MIN_PLAYERS, MAX_PLAYERS + 1)):
    failures.append(f"unexpected supported_player_counts={summary.get('supported_player_counts')}")
if int(summary.get('maximum_frontier_rows', MAX_FRONTIER_ROWS + 1)) > MAX_FRONTIER_ROWS:
    failures.append('materialized frontier exceeded the configured row ceiling')

for label in ('advantage_iteration_ranges', 'strategy_iteration_ranges'):
    ranges = summary.get(label, [])
    if len(ranges) != MAX_PLAYERS or any(
        retained is None or int(retained[1]) != TARGET_ITERATIONS for retained in ranges
    ):
        failures.append(f'{label} does not retain the final iteration for all seats: {ranges}')

policy_payload = torch.load(LOCAL_POLICY, map_location='cpu', weights_only=False)
if policy_payload.get('algorithm') != 'deep_cfr_average_strategy':
    failures.append(f"unexpected policy algorithm={policy_payload.get('algorithm')}")
for seat, network_state in enumerate(policy_payload.get('strategy_networks', [])):
    for name, tensor in network_state.items():
        if torch.is_tensor(tensor) and not bool(torch.isfinite(tensor).all().item()):
            failures.append(f'non-finite policy tensor at seat {seat}: {name}')

from poker_arena import Table, TableConfig, TorchPolicyBot

try:
    smoke_bot = TorchPolicyBot.from_checkpoint(LOCAL_POLICY, device='cpu')
    for seats in range(MIN_PLAYERS, MAX_PLAYERS + 1):
        table = Table(TableConfig(seats, SMALL_BLIND, BIG_BLIND, [STARTING_STACK] * seats, seed=1000 + seats))
        state = table.start_hand()
        actor = state.current_actor
        action = smoke_bot.choose_action(state.player_view(actor), state.legal_actions(actor))
        table.apply(action)
except Exception as error:
    failures.append(f'engine inference smoke test failed: {type(error).__name__}: {error}')

INTEGRITY_PASS = not failures
integrity_report = {
    'integrity_pass': INTEGRITY_PASS,
    'failures': failures,
    'iterations': summary.get('iterations'),
    'traversals': summary.get('traversals'),
    'player_count_traversals': coverage,
    'depth_limit_rollouts': summary.get('depth_limit_rollouts'),
    'latest_iteration_retained': summary.get('latest_iteration_retained'),
    'maximum_frontier_rows': summary.get('maximum_frontier_rows'),
    'maximum_projected_frontier_rows': summary.get('maximum_projected_frontier_rows'),
    'frontier_chunk_splits': summary.get('frontier_chunk_splits'),
    'policy_bytes': LOCAL_POLICY.stat().st_size,
}
LOCAL_INTEGRITY = LOCAL_RUN_DIR / f'{RUN_NAME}_integrity.json'
DRIVE_INTEGRITY = DRIVE_RUN_DIR / LOCAL_INTEGRITY.name
LOCAL_INTEGRITY.write_text(json.dumps(integrity_report, indent=2), encoding='utf-8')
atomic_copy_to_drive(LOCAL_INTEGRITY, DRIVE_INTEGRITY)
print(json.dumps(integrity_report, indent=2))
if not INTEGRITY_PASS:
    write_drive_progress(TARGET_ITERATIONS, 'completed_failed_integrity')
    raise RuntimeError('Training completed, but integrity gates failed. The policy will not be packaged for deployment.')
write_drive_progress(TARGET_ITERATIONS, 'completed_integrity_passed')


## Build and automatically download the engine bundle

Only the inference policy `.pt` is required by the poker GUI. The ZIP also contains the training summary, integrity report, training log, checksums, and the exact Windows launcher argument. The multi-gigabyte resumable snapshot stays in Google Drive.

In [ ]:
import zipfile
from google.colab import files

if not INTEGRITY_PASS:
    raise RuntimeError('Refusing to download an engine bundle because integrity gates did not pass.')

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

manifest = {
    'engine_checkpoint': LOCAL_POLICY.name,
    'engine_checkpoint_sha256': sha256_file(LOCAL_POLICY),
    'engine_requires_only_checkpoint_pt': True,
    'supported_player_counts': list(range(MIN_PLAYERS, MAX_PLAYERS + 1)),
    'starting_stack_big_blinds': STARTING_STACK / BIG_BLIND,
    'integrity_pass': True,
    'resume_snapshot_in_drive': str(DRIVE_SNAPSHOT),
    'windows_example': (
        'powershell -ExecutionPolicy Bypass -File .\\scripts\\run_poker_arena.ps1 '
        f'-ModelPath .\\runs\\{LOCAL_POLICY.name} -Device cuda'
    ),
}
LOCAL_MANIFEST = LOCAL_RUN_DIR / f'{RUN_NAME}_manifest.json'
LOCAL_MANIFEST.write_text(json.dumps(manifest, indent=2), encoding='utf-8')

deployment_readme = LOCAL_RUN_DIR / 'DEPLOYMENT.txt'
deployment_readme.write_text(
    'Poker Arena engine bundle\n'
    '=========================\n\n'
    f'1. Extract and copy {LOCAL_POLICY.name} into the project runs folder.\n'
    '2. From the project root, launch the GUI with:\n\n'
    f"   {manifest['windows_example']}\n\n"
    'The GUI loader recognizes this Deep CFR average-strategy checkpoint automatically.\n'
    'No training snapshot or code change is required for play. The large snapshot in Drive is only for resuming training.\n',
    encoding='utf-8',
)

bundle_path = LOCAL_RUN_DIR / f'{RUN_NAME}_engine_bundle.zip'
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_STORED) as archive:
    for artifact in (LOCAL_POLICY, LOCAL_SUMMARY, LOCAL_INTEGRITY, LOCAL_MANIFEST, deployment_readme, LOCAL_LOG):
        if artifact.is_file():
            archive.write(artifact, arcname=artifact.name)

DRIVE_BUNDLE = DRIVE_RUN_DIR / bundle_path.name
atomic_copy_to_drive(bundle_path, DRIVE_BUNDLE)
print(f'Engine bundle saved to Drive: {DRIVE_BUNDLE}')
print(f'Starting browser download: {bundle_path.name} ({bundle_path.stat().st_size / 1024**2:.1f} MiB)')
files.download(str(bundle_path))
